# Assignment 1: Tabular Reinforcement Learning

## Overview

In this assignment, you will build the foundations of reinforcement learning from scratch. You will
implement a GridWorld environment, solve it exactly with Dynamic Programming, then learn to solve
it from experience using Q-Learning and SARSA.

Every algorithm you encounter later -- DQN, PPO, RLHF -- is a descendant of the tabular methods
you build here. Master the tabular case and the deep RL extensions will be natural.

### Learning Objectives
- Implement a GridWorld environment from scratch
- Solve MDPs exactly with Value Iteration and Policy Iteration
- Implement Q-Learning (off-policy) and SARSA (on-policy)
- Understand the exploration-exploitation tradeoff via epsilon schedules
- Demonstrate the on-policy vs off-policy difference on CliffWorld

### Key Equations

**Bellman Optimality Equation:**

$$V^*(s) = \max_a \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V^*(s') \right]$$

**Q-Learning Update:**

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

**SARSA Update:**

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma Q(s',a') - Q(s,a) \right]$$

**Estimated time:** 10-14 hours

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from collections import defaultdict
import sys
sys.path.insert(0, '../../')

from shared_utils.common import set_seed

set_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

### Helper Functions

In [ ]:
# Action mappings
ACTIONS = {0: 'UP', 1: 'DOWN', 2: 'LEFT', 3: 'RIGHT'}
ACTION_DELTAS = {
    0: (-1, 0),  # UP
    1: (1, 0),   # DOWN
    2: (0, -1),  # LEFT
    3: (0, 1),   # RIGHT
}
ARROW_SYMBOLS = {0: '\u2191', 1: '\u2193', 2: '\u2190', 3: '\u2192'}  # up, down, left, right


def plot_value_function(V, grid_size, title="Value Function"):
    """Plot the value function as a heatmap on the grid.
    
    Args:
        V: Value function array of shape (n_states,)
        grid_size: Tuple (rows, cols) or int for square grids
        title: Plot title
    """
    if isinstance(grid_size, int):
        grid_size = (grid_size, grid_size)
    rows, cols = grid_size
    V_grid = V.reshape(rows, cols)
    
    fig, ax = plt.subplots(figsize=(cols + 2, rows + 2))
    im = ax.imshow(V_grid, cmap='RdYlGn', interpolation='nearest')
    
    # Add value annotations
    for i in range(rows):
        for j in range(cols):
            ax.text(j, i, f"{V_grid[i, j]:.1f}", ha='center', va='center',
                    fontsize=10, fontweight='bold')
    
    ax.set_title(title, fontsize=14)
    ax.set_xticks(range(cols))
    ax.set_yticks(range(rows))
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


def plot_policy(policy, grid_size, title="Policy"):
    """Plot the policy as arrows on the grid.
    
    Args:
        policy: Policy array of shape (n_states,) with action indices
        grid_size: Tuple (rows, cols) or int for square grids
        title: Plot title
    """
    if isinstance(grid_size, int):
        grid_size = (grid_size, grid_size)
    rows, cols = grid_size
    
    fig, ax = plt.subplots(figsize=(cols + 1, rows + 1))
    ax.set_xlim(-0.5, cols - 0.5)
    ax.set_ylim(rows - 0.5, -0.5)
    
    for state in range(rows * cols):
        r, c = state // cols, state % cols
        action = policy[state]
        ax.text(c, r, ARROW_SYMBOLS[action], ha='center', va='center', fontsize=20)
    
    # Draw grid lines
    for i in range(rows + 1):
        ax.axhline(y=i - 0.5, color='black', linewidth=0.5)
    for j in range(cols + 1):
        ax.axvline(x=j - 0.5, color='black', linewidth=0.5)
    
    # Mark start and goal
    ax.add_patch(plt.Rectangle((-0.5, -0.5), 1, 1, fill=True, color='lightblue', alpha=0.5))
    ax.add_patch(plt.Rectangle((cols - 1 - 0.5, rows - 1 - 0.5), 1, 1, fill=True, color='lightgreen', alpha=0.5))
    
    ax.set_title(title, fontsize=14)
    ax.set_xticks(range(cols))
    ax.set_yticks(range(rows))
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()


def plot_learning_curves(rewards_dict, window=100, title="Learning Curves"):
    """Plot smoothed learning curves for multiple agents.
    
    Args:
        rewards_dict: Dict mapping agent names to lists of per-episode rewards
        window: Smoothing window size
        title: Plot title
    """
    fig, ax = plt.subplots(figsize=(10, 5))
    
    for name, rewards in rewards_dict.items():
        # Compute running average
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(smoothed, label=name)
    
    ax.set_xlabel('Episode')
    ax.set_ylabel(f'Reward (rolling avg, window={window})')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print("Helper functions loaded.")

---
## Part 1: Implement a GridWorld Environment

### 1.1 The Environment

Build a GridWorld environment from scratch (do NOT use Gymnasium).

**Specification:**
- Grid size: configurable (default 5x5)
- States: each cell is a state, identified by flat index
- Actions: 4 discrete -- UP (0), DOWN (1), LEFT (2), RIGHT (3)
- Transitions: deterministic. Moving into a wall keeps the agent in place.
- Start state: top-left corner (0, 0)
- Goal state: bottom-right corner
- Reward: -1 per step, 0 at the goal
- Terminal: episode ends when agent reaches the goal

In [ ]:
class GridWorld:
    """A simple GridWorld environment."""
    
    def __init__(self, size: int = 5, stochastic: bool = False):
        """Initialize GridWorld.
        
        Args:
            size: Grid dimension (size x size)
            stochastic: If True, actions succeed with probability 0.8,
                       with probability 0.2 a random action is taken instead
        """
        self.size = size
        self.n_states = size * size
        self.n_actions = 4
        self.stochastic = stochastic
        self.start_state = 0
        self.goal_state = self.n_states - 1
        self.current_state = self.start_state
    
    def reset(self) -> int:
        """Reset to start state. Return initial state."""
        # YOUR CODE HERE
        raise NotImplementedError("Implement GridWorld.reset")
    
    def step(self, action: int) -> tuple:
        """Take action. Return (next_state, reward, done)."""
        # YOUR CODE HERE
        # 1. If stochastic: with prob 0.2, replace action with random action
        # 2. Compute next position based on action
        # 3. If next position is out of bounds (wall), stay in place
        # 4. Return (next_state, reward, done)
        #    - reward = 0 if at goal, -1 otherwise
        #    - done = True if at goal
        raise NotImplementedError("Implement GridWorld.step")
    
    def get_transition_prob(self, state: int, action: int) -> list:
        """Return list of (next_state, reward, probability) tuples.
        
        Needed for Dynamic Programming methods.
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement GridWorld.get_transition_prob")
    
    def state_to_rowcol(self, state: int) -> tuple:
        """Convert flat state index to (row, col)."""
        return state // self.size, state % self.size
    
    def rowcol_to_state(self, row: int, col: int) -> int:
        """Convert (row, col) to flat state index."""
        return row * self.size + col
    
    def render(self):
        """Print the grid with agent position marked."""
        for r in range(self.size):
            row_str = ""
            for c in range(self.size):
                state = self.rowcol_to_state(r, c)
                if state == self.current_state:
                    row_str += " A "
                elif state == self.goal_state:
                    row_str += " G "
                else:
                    row_str += " . "
            print(row_str)

### Test: GridWorld Environment

In [ ]:
def test_gridworld():
    """Verify GridWorld behavior."""
    env = GridWorld(size=5)
    
    # Test reset
    state = env.reset()
    assert state == 0, f"Reset should return state 0, got {state}"
    print("PASSED: Reset returns state 0")
    
    # Test step down from state 0
    next_state, reward, done = env.step(1)  # DOWN
    assert next_state == 5, f"Down from state 0 should give state 5, got {next_state}"
    assert reward == -1, f"Non-goal reward should be -1, got {reward}"
    assert done == False, "Should not be done"
    print("PASSED: Step down from (0,0) -> (1,0)")
    
    # Test wall collision
    env.reset()
    next_state, reward, done = env.step(0)  # UP from (0,0)
    assert next_state == 0, f"Up from (0,0) should stay at 0, got {next_state}"
    print("PASSED: Wall collision keeps agent in place")
    
    # Test reaching goal
    env.current_state = env.goal_state - 1  # One step left of goal (bottom row)
    next_state, reward, done = env.step(3)  # RIGHT
    assert next_state == env.goal_state, f"Should reach goal, got state {next_state}"
    assert reward == 0, f"Goal reward should be 0, got {reward}"
    assert done == True, "Should be done at goal"
    print("PASSED: Reaching goal returns reward=0, done=True")
    
    # Display the grid
    env.reset()
    print("\nInitial grid:")
    env.render()
    
    print("\nAll GridWorld tests passed!")

test_gridworld()

---
## Part 2: Dynamic Programming -- Solve the GridWorld Exactly

### 2.1 Value Iteration

Iterate the Bellman optimality equation until convergence:

$$V(s) \leftarrow \max_a \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V(s') \right]$$

In [ ]:
def value_iteration(env, gamma: float = 0.99, theta: float = 1e-8) -> tuple:
    """Find optimal V* and policy using Value Iteration.
    
    Args:
        env: GridWorld environment
        gamma: Discount factor
        theta: Convergence threshold
        
    Returns:
        V: Optimal value function, array of shape (n_states,)
        policy: Optimal policy, array of shape (n_states,) mapping state -> action
        n_iterations: Number of iterations to converge
    """
    # YOUR CODE HERE
    # 1. Initialize V(s) = 0 for all states
    # 2. Repeat:
    #    a. For each state, compute max_a sum of P(s'|s,a)[R + gamma*V(s')]
    #    b. Track maximum change (delta)
    #    c. Stop when delta < theta
    # 3. Extract greedy policy from converged V
    raise NotImplementedError("Implement value_iteration")

In [ ]:
# Run Value Iteration on deterministic GridWorld
env = GridWorld(size=5, stochastic=False)
V_star, policy_star, n_iter = value_iteration(env)
print(f"Value Iteration converged in {n_iter} iterations")

plot_value_function(V_star, 5, title="Optimal Value Function V* (Deterministic)")
plot_policy(policy_star, 5, title="Optimal Policy (Deterministic)")

In [ ]:
# Run on stochastic GridWorld
env_stoch = GridWorld(size=5, stochastic=True)
V_stoch, policy_stoch, n_iter_stoch = value_iteration(env_stoch)
print(f"Value Iteration (stochastic) converged in {n_iter_stoch} iterations")

plot_value_function(V_stoch, 5, title="Optimal Value Function V* (Stochastic)")
plot_policy(policy_stoch, 5, title="Optimal Policy (Stochastic)")

### 2.2 Policy Iteration

In [ ]:
def policy_iteration(env, gamma: float = 0.99, theta: float = 1e-8) -> tuple:
    """Find optimal V* and policy using Policy Iteration.
    
    Args:
        env: GridWorld environment
        gamma: Discount factor
        theta: Convergence threshold for policy evaluation
        
    Returns:
        V: Optimal value function, array of shape (n_states,)
        policy: Optimal policy, array of shape (n_states,)
        n_iterations: Number of policy improvement steps
    """
    # YOUR CODE HERE
    # 1. Initialize random policy
    # 2. Repeat:
    #    a. Policy Evaluation: solve V^pi iteratively
    #    b. Policy Improvement: make policy greedy w.r.t. V^pi
    #    c. Stop when policy is stable
    raise NotImplementedError("Implement policy_iteration")

In [ ]:
# Run Policy Iteration and compare with Value Iteration
env = GridWorld(size=5, stochastic=False)
V_pi, policy_pi, n_pi_iter = policy_iteration(env)
print(f"Policy Iteration converged in {n_pi_iter} policy improvement steps")

# Verify both methods produce the same policy
assert np.array_equal(policy_star, policy_pi), "Value Iteration and Policy Iteration should agree!"
print("PASSED: Both methods produce the same optimal policy")
print(f"Max V difference: {np.max(np.abs(V_star - V_pi)):.2e}")

---
## Part 3: Q-Learning -- Learn from Experience

### 3.1 Q-Learning Agent

In [ ]:
class QLearningAgent:
    """Tabular Q-Learning agent."""
    
    def __init__(
        self,
        n_states: int,
        n_actions: int,
        alpha: float = 0.1,
        gamma: float = 0.99,
        epsilon: float = 1.0,
        epsilon_min: float = 0.01,
        epsilon_decay: float = 0.995
    ):
        """Initialize Q-Learning agent.
        
        Args:
            n_states: Number of states
            n_actions: Number of actions
            alpha: Learning rate
            gamma: Discount factor
            epsilon: Initial exploration rate
            epsilon_min: Minimum exploration rate
            epsilon_decay: Epsilon decay factor per episode
        """
        # YOUR CODE HERE
        # Initialize Q-table to zeros: shape (n_states, n_actions)
        raise NotImplementedError("Initialize QLearningAgent")
    
    def select_action(self, state: int) -> int:
        """Epsilon-greedy action selection."""
        # YOUR CODE HERE
        raise NotImplementedError("Implement select_action")
    
    def update(self, state: int, action: int, reward: float, next_state: int, done: bool):
        """Q-Learning update: Q(s,a) <- Q(s,a) + alpha * [r + gamma * max_a' Q(s',a') - Q(s,a)]"""
        # YOUR CODE HERE
        raise NotImplementedError("Implement Q-Learning update")
    
    def decay_epsilon(self):
        """Decay exploration rate after each episode."""
        # YOUR CODE HERE
        raise NotImplementedError("Implement epsilon decay")
    
    def get_policy(self) -> np.ndarray:
        """Extract greedy policy from Q-table."""
        return np.argmax(self.Q, axis=1)

### 3.2 Training Loop

In [ ]:
def train_q_learning(env, agent, n_episodes: int = 5000) -> tuple:
    """Train the Q-Learning agent.
    
    Args:
        env: GridWorld environment
        agent: QLearningAgent instance
        n_episodes: Number of episodes to train
        
    Returns:
        rewards_per_episode: List of total rewards for each episode
        steps_per_episode: List of steps taken per episode
    """
    # YOUR CODE HERE
    # For each episode:
    # 1. Reset environment
    # 2. Loop until done (or max steps):
    #    a. Select action (epsilon-greedy)
    #    b. Take step in environment
    #    c. Update Q-table
    # 3. Decay epsilon
    # 4. Track rewards and steps
    raise NotImplementedError("Implement train_q_learning")

In [ ]:
# Train Q-Learning agent
env = GridWorld(size=5, stochastic=False)
agent = QLearningAgent(env.n_states, env.n_actions)
rewards, steps = train_q_learning(env, agent, n_episodes=5000)

# Compare learned policy to optimal
learned_policy = agent.get_policy()
print("Learned policy matches optimal:", np.array_equal(learned_policy, policy_star))

plot_policy(learned_policy, 5, title="Q-Learning Learned Policy")

### 3.3 Epsilon Decay Analysis

Compare three epsilon schedules:
1. Constant epsilon = 0.1
2. Linear decay from 1.0 to 0.01
3. Exponential decay: epsilon *= 0.995

In [ ]:
# Train with different epsilon schedules
n_episodes = 5000
epsilon_results = {}

# Schedule 1: Constant epsilon = 0.1
set_seed(42)
env = GridWorld(size=5)
agent_const = QLearningAgent(env.n_states, env.n_actions, epsilon=0.1, epsilon_min=0.1, epsilon_decay=1.0)
rewards_const, _ = train_q_learning(env, agent_const, n_episodes)
epsilon_results['Constant (0.1)'] = rewards_const

# Schedule 2: Linear decay
# YOUR CODE HERE

# Schedule 3: Exponential decay
set_seed(42)
env = GridWorld(size=5)
agent_exp = QLearningAgent(env.n_states, env.n_actions, epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.995)
rewards_exp, _ = train_q_learning(env, agent_exp, n_episodes)
epsilon_results['Exponential (0.995)'] = rewards_exp

plot_learning_curves(epsilon_results, window=100, title='Q-Learning: Epsilon Schedule Comparison')

---
## Part 4: SARSA -- On-Policy Learning

### 4.1 SARSA Agent

The key difference: SARSA uses $Q(s', a')$ where $a'$ is the action actually taken,
while Q-Learning uses $\max_{a'} Q(s', a')$.

In [ ]:
class SARSAAgent:
    """Tabular SARSA agent (on-policy)."""
    
    def __init__(
        self,
        n_states: int,
        n_actions: int,
        alpha: float = 0.1,
        gamma: float = 0.99,
        epsilon: float = 1.0,
        epsilon_min: float = 0.01,
        epsilon_decay: float = 0.995
    ):
        # YOUR CODE HERE
        raise NotImplementedError("Initialize SARSAAgent")
    
    def select_action(self, state: int) -> int:
        """Epsilon-greedy action selection."""
        # YOUR CODE HERE
        raise NotImplementedError("Implement select_action")
    
    def update(self, state: int, action: int, reward: float, next_state: int, next_action: int, done: bool):
        """SARSA update: Q(s,a) <- Q(s,a) + alpha * [r + gamma * Q(s',a') - Q(s,a)]"""
        # YOUR CODE HERE
        raise NotImplementedError("Implement SARSA update")
    
    def decay_epsilon(self):
        """Decay exploration rate."""
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def get_policy(self) -> np.ndarray:
        """Extract greedy policy from Q-table."""
        return np.argmax(self.Q, axis=1)

### 4.2 The Cliff-Walking Comparison

Implement a CliffWorld environment:
- Grid: 4 rows x 12 columns
- Start: bottom-left (3, 0)
- Goal: bottom-right (3, 11)
- Cliff: cells (3, 1) through (3, 10) -- stepping on cliff gives -100 reward and resets to start
- All other steps: -1 reward

In [ ]:
class CliffWorld:
    """CliffWorld environment for comparing Q-Learning and SARSA."""
    
    def __init__(self):
        self.rows = 4
        self.cols = 12
        self.n_states = self.rows * self.cols
        self.n_actions = 4
        self.start_state = self.rowcol_to_state(3, 0)
        self.goal_state = self.rowcol_to_state(3, 11)
        # Cliff: bottom row, columns 1 through 10
        self.cliff_states = set(
            self.rowcol_to_state(3, c) for c in range(1, 11)
        )
        self.current_state = self.start_state
    
    def rowcol_to_state(self, row: int, col: int) -> int:
        return row * self.cols + col
    
    def state_to_rowcol(self, state: int) -> tuple:
        return state // self.cols, state % self.cols
    
    def reset(self) -> int:
        # YOUR CODE HERE
        raise NotImplementedError("Implement CliffWorld.reset")
    
    def step(self, action: int) -> tuple:
        """Take action. Return (next_state, reward, done).
        
        Cliff: reward = -100, reset to start.
        Goal: reward = 0, done = True.
        Other: reward = -1.
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement CliffWorld.step")

In [ ]:
def train_sarsa(env, agent, n_episodes: int = 5000) -> tuple:
    """Train the SARSA agent.
    
    Note: SARSA needs next_action in update, so the loop structure is slightly different.
    """
    # YOUR CODE HERE
    # For each episode:
    # 1. Reset, select first action
    # 2. Loop until done:
    #    a. Take step with current action
    #    b. Select NEXT action
    #    c. Update Q(s,a) using (s, a, r, s', a')
    #    d. s <- s', a <- a'
    # 3. Decay epsilon
    raise NotImplementedError("Implement train_sarsa")

In [ ]:
# Train Q-Learning and SARSA on CliffWorld with constant epsilon = 0.1
n_episodes = 5000
cliff_env = CliffWorld()

# Q-Learning
set_seed(42)
q_agent = QLearningAgent(cliff_env.n_states, cliff_env.n_actions,
                         alpha=0.1, gamma=0.99, epsilon=0.1, epsilon_min=0.1, epsilon_decay=1.0)
q_rewards, _ = train_q_learning(cliff_env, q_agent, n_episodes)

# SARSA
set_seed(42)
cliff_env = CliffWorld()
sarsa_agent = SARSAAgent(cliff_env.n_states, cliff_env.n_actions,
                         alpha=0.1, gamma=0.99, epsilon=0.1, epsilon_min=0.1, epsilon_decay=1.0)
sarsa_rewards, _ = train_sarsa(cliff_env, sarsa_agent, n_episodes)

# Plot comparison
plot_learning_curves(
    {'Q-Learning': q_rewards, 'SARSA': sarsa_rewards},
    window=100,
    title='CliffWorld: Q-Learning vs SARSA'
)

In [ ]:
# Visualize the learned policies on CliffWorld
q_policy = q_agent.get_policy()
sarsa_policy = sarsa_agent.get_policy()

plot_policy(q_policy, (4, 12), title="Q-Learning Policy on CliffWorld (optimal path, near cliff)")
plot_policy(sarsa_policy, (4, 12), title="SARSA Policy on CliffWorld (safe path, away from cliff)")

---
## Part 5: Analysis and Visualization

### 5.1 Learned Value Function Heatmap

In [ ]:
# Compare Q-Learning's learned V to DP's V*
# V(s) = max_a Q(s,a)
V_learned = np.max(agent.Q, axis=1)  # From Part 3 agent trained on GridWorld

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# DP value function
im1 = ax1.imshow(V_star.reshape(5, 5), cmap='RdYlGn')
ax1.set_title('Value Iteration V*')
for i in range(5):
    for j in range(5):
        ax1.text(j, i, f"{V_star[i*5+j]:.1f}", ha='center', va='center', fontsize=9)
plt.colorbar(im1, ax=ax1, shrink=0.8)

# Q-Learning value function
im2 = ax2.imshow(V_learned.reshape(5, 5), cmap='RdYlGn')
ax2.set_title('Q-Learning V (learned)')
for i in range(5):
    for j in range(5):
        ax2.text(j, i, f"{V_learned[i*5+j]:.1f}", ha='center', va='center', fontsize=9)
plt.colorbar(im2, ax=ax2, shrink=0.8)

plt.suptitle('Value Function Comparison: DP vs Q-Learning', fontsize=14)
plt.tight_layout()
plt.show()

### 5.3 Learning Dynamics

In [ ]:
# Plot running average reward, epsilon, and max Q-value over training
# YOUR CODE HERE
# 1. Running average reward (window=100)
# 2. Epsilon over episodes
# 3. Max Q-value over episodes

### 5.4 Written Analysis

Write a 300-500 word analysis addressing:
1. Do Q-Learning and Value Iteration converge to the same policy? Why or why not?
2. How does the epsilon decay schedule affect convergence speed and final performance?
3. Why do Q-Learning and SARSA learn different policies on CliffWorld? Which would you prefer in a safety-critical application?
4. What happens when you increase/decrease alpha?
5. How does gamma affect the learned policy?

**Your analysis here:**

YOUR ANSWER HERE

---
## Stretch Goals (Optional)

1. **TD(lambda)**: Implement Q-Learning with eligibility traces. Compare for lambda = 0, 0.5, 0.9, 1.0.
2. **Taxi-v3**: Apply Q-Learning to Gymnasium's Taxi-v3 (500 states).
3. **Expected SARSA**: Use expected Q-value under the policy instead of a single sample.
4. **Q-value convergence animation**: Animate how Q-values evolve over training.

In [ ]:
# Stretch goal workspace

# YOUR STRETCH GOAL CODE HERE